# LLM APIs, Reasoning Flows & Prompt-to-Action Pipelines

**Track:** Applied Agent Engineering Foundations  
**Module:** LLM Foundations  
**Environment:** SageMaker Jupyter Notebook  
**Audience:** Software engineers building enterprise AI workflows on AWS

## What learners will learn
By the end of this lab, learners will be able to:
1. Connect securely from SageMaker to Amazon Bedrock.
2. Discover available Bedrock foundation models from code.
3. Use Bedrock LLM APIs for prompt experiments and structured generation.
4. Build a prompt-to-action pipeline with tool/function-calling style behavior.
5. Validate model output before execution.
6. Manage context windows using chunking and budget checks.
7. Add safe error handling and run logging for enterprise usage.

## Instructor flow
- Part A: Secure setup and model discovery
- Part B: Prompt styles and reasoning flows
- Part C: Prompt-to-action pipeline and tool/function calling pattern
- Part D: Context window management
- Part E: Error handling, validation, and mini-hack

## Before you run

This notebook assumes:
- you are running inside **SageMaker Jupyter Notebook**
- your notebook has an **execution role** with Bedrock and S3 read permissions
- Bedrock model access is already enabled for the models you want to test
- your learning files are already uploaded to **S3**
- you are **reading** from S3 only in this notebook; no S3 write-back is required

### Design choice for this lab
This notebook teaches a **model-agnostic prompt-to-action pipeline**.  
That means learners first understand:
- how to call an LLM API
- how to get structured output
- how to validate it
- how to safely call backend functions

Later, the same pattern can be extended to native tool use, agents, workflows, or orchestration frameworks.

In [1]:
# Uncomment only if your environment is missing any package
# %pip install -q boto3 pandas

import json
import re
import textwrap
from typing import Any, Dict, List, Optional

import boto3
import pandas as pd
from botocore.exceptions import ClientError

AWS_REGION = boto3.Session().region_name or "us-east-1"

# Bedrock clients
bedrock = boto3.client("bedrock", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# Other AWS clients we will use in this notebook
s3 = boto3.client("s3", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)

# Choose one classroom model that supports text generation through Converse.
# Change this only if your account has access to a different model.
BEDROCK_TEXT_MODEL = "amazon.nova-lite-v1:0"

# Optional allowlist for secure endpoint usage in classroom demos
APPROVED_MODEL_IDS = {BEDROCK_TEXT_MODEL}

print("AWS Region:", AWS_REGION)
print("Default text model:", BEDROCK_TEXT_MODEL)
print("Caller ARN:", sts.get_caller_identity()["Arn"])

AWS Region: us-east-1
Default text model: amazon.nova-lite-v1:0
Caller ARN: arn:aws:sts::502761807248:assumed-role/SageMakerExecutionRole-S3Controlled/SageMaker


## Step 1 — Show available models in Bedrock

**Goal:** Let learners see what Bedrock exposes in the current region.

**Discuss:**
- model access can vary by account and region
- some models support chat, some embeddings, some image or multimodal inputs
- classroom code should **discover** available models instead of relying on memory

In [3]:
def list_bedrock_models() -> pd.DataFrame:
    response = bedrock.list_foundation_models()
    rows = []

    for item in response.get("modelSummaries", []):
        rows.append({
            "provider": item.get("providerName"),
            "model_id": item.get("modelId"),
            "model_name": item.get("modelName"),
            "input_modalities": ", ".join(item.get("inputModalities", [])),
            "output_modalities": ", ".join(item.get("outputModalities", [])),
            "streaming": item.get("responseStreamingSupported"),
            "customizations": ", ".join(item.get("customizationsSupported", [])),
            "inference_types": ", ".join(item.get("inferenceTypesSupported", [])),
        })

    df = pd.DataFrame(rows).sort_values(["provider", "model_id"]).reset_index(drop=True)
    return df

try:
    models_df = list_bedrock_models()
    with pd.option_context('display.max_rows', 250):
        display(models_df)
except ClientError as e:
    print("Could not list models. Check Bedrock permissions and regional access.")
    print(e)

,provider,model_id,model_name,input_modalities,output_modalities,streaming,customizations,inference_types
0,AI21 Labs,ai21.jamba-1-5-large-v1:0,Jamba 1.5 Large,TEXT,TEXT,True,,ON_DEMAND
1,AI21 Labs,ai21.jamba-1-5-mini-v1:0,Jamba 1.5 Mini,TEXT,TEXT,True,,ON_DEMAND
2,Amazon,amazon.nova-2-lite-v1:0,Nova 2 Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,INFERENCE_PROFILE
3,Amazon,amazon.nova-2-lite-v1:0:256k,Nova 2 Lite,"TEXT, IMAGE, VIDEO",TEXT,True,FINE_TUNING,PROVISIONED
4,Amazon,amazon.nova-2-multimodal-embeddings-v1:0,Amazon Nova Multimodal Embeddings,"TEXT, IMAGE, AUDIO, VIDEO",EMBEDDING,False,,ON_DEMAND
5,Amazon,amazon.nova-2-sonic-v1:0,Nova 2 Sonic,SPEECH,"SPEECH, TEXT",True,,ON_DEMAND
6,Amazon,amazon.nova-canvas-v1:0,Nova Canvas,"TEXT, IMAGE",IMAGE,False,FINE_TUNING,"ON_DEMAND, PROVISIONED"
7,Amazon,amazon.nova-lite-v1:0,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,"ON_DEMAND, INFERENCE_PROFILE"
8,Amazon,amazon.nova-lite-v1:0:24k,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,PROVISIONED
9,Amazon,amazon.nova-lite-v1:0:300k,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True,"FINE_TUNING, DISTILLATION",PROVISIONED


## Step 2 — Secure endpoint usage and minimal smoke test

**Goal:** Confirm that SageMaker can call Bedrock safely.

**Secure usage principles:**
- do **not** hardcode keys in notebooks
- use the **SageMaker execution role**
- restrict model IDs through config or an allowlist
- start with a **small, cheap** smoke test
- fail early if the model ID is not approved

**Why this matters:**  
In enterprise settings, the first failure should happen **before** a costly or unsafe request is sent.

In [4]:
def validate_model_id(model_id: str) -> None:
    if APPROVED_MODEL_IDS and model_id not in APPROVED_MODEL_IDS:
        raise ValueError(f"Model '{model_id}' is not in the approved classroom allowlist.")

def ask_llm(user_text: str,
            system_text: str = "You are a helpful enterprise AI assistant.",
            model_id: str = BEDROCK_TEXT_MODEL,
            max_tokens: int = 250,
            temperature: float = 0.2) -> str:
    validate_model_id(model_id)

    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_text}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_text}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    )
    return response["output"]["message"]["content"][0]["text"]

print(ask_llm("In two lines, explain what an LLM API call is."))

An LLM API call is a request made to a language model's interface to generate text based on provided input. It allows developers to integrate natural language processing capabilities into their applications.


## Step 3 — Prompt styles and reasoning flows

We will test three common patterns:
1. simple instruction
2. role-based instruction
3. structured reasoning flow

**Explain to learners:**  
Better prompts are not just longer prompts.  
Better prompts make the model's job easier by clarifying:
- role
- task
- output shape
- constraints
- reasoning path

In [5]:
prompts = {
    "simple_instruction": (
        "Summarize why validation matters in enterprise AI systems."
    ),
    "role_based": (
        "You are a senior platform architect. "
        "Summarize why validation matters in enterprise AI systems "
        "for backend engineers."
    ),
    "reasoning_flow": (
        "You are helping an engineering team design a safe AI workflow. "
        "First give 3 short reasoning steps. "
        "Then give a final recommendation. "
        "Return in this format:\n"
        "Reasoning Steps:\n"
        "- ...\n"
        "- ...\n"
        "- ...\n"
        "Final Recommendation:\n"
        "..."
    )
}

for name, prompt in prompts.items():
    print("=" * 90)
    print("PROMPT STYLE:", name)
    print(ask_llm(prompt))
    print()

PROMPT STYLE: simple_instruction
Validation is crucial in enterprise AI systems for several reasons:

1. **Accuracy and Reliability**: Validation ensures that AI models produce accurate and reliable results, which is essential for making informed business decisions.

2. **Risk Management**: Validating AI systems helps identify and mitigate potential risks, such as biases, errors, and security vulnerabilities, thereby protecting the organization from financial and reputational damage.

3. **Compliance**: Many industries are subject to regulatory requirements. Validating AI systems ensures compliance with legal and industry standards, avoiding potential penalties.

4. **User Trust**: Validated AI systems build trust among users and stakeholders, as they can be confident in the system's performance and integrity.

5. **Performance Optimization**: Validation processes help in fine-tuning models to improve performance, ensuring they meet the desired business objectives.

6. **Cost Efficienc

## Reflection checkpoint

Discuss these questions before moving on:
- Which output is easiest to trust?
- Which output is easiest to parse downstream?
- Which output is easiest to test automatically?
- Which output best supports a prompt-to-action workflow?

## Step 4 — Build a prompt-to-action pipeline

A production-safe workflow often looks like this:
1. understand the request
2. convert it into a structured plan
3. validate the plan
4. execute only the allowed action
5. call a controlled backend function
6. log the result

This is the core **tool/function calling pattern** we want learners to understand.

### Important note
This notebook uses a **model-agnostic structured JSON plan**.  
That keeps the concept simple and portable across models.

In [6]:
ACTION_PLANNER_PROMPT = '''
You are an action planner for an enterprise assistant.

Return valid JSON only.
Use this schema:
{
  "intent": "<one short label>",
  "needs_tool": true or false,
  "tool_name": "<tool or none>",
  "arguments": { ... },
  "reason_for_tool": "<one sentence>"
}

Allowed tool names:
- create_ticket
- lookup_user_record
- none

Examples:
User: "Create an IT ticket for VPN access issue"
Output:
{"intent":"create_it_ticket","needs_tool":true,"tool_name":"create_ticket","arguments":{"category":"IT Support","summary":"VPN access issue"},"reason_for_tool":"A ticket must be created in the backend system."}

User: "Find the team and location for ananya"
Output:
{"intent":"lookup_user_record","needs_tool":true,"tool_name":"lookup_user_record","arguments":{"user_name":"ananya"},"reason_for_tool":"The answer should come from the CSV file loaded from S3."}
'''

def extract_json_object(text: str) -> dict:
    text = text.strip()

    # direct JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # JSON inside markdown fences
    fenced = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if fenced:
        return json.loads(fenced.group(1))

    # first JSON object fallback
    match = re.search(r"(\{.*\})", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))

    raise json.JSONDecodeError("No valid JSON object found", text, 0)

def plan_action(user_request: str) -> dict:
    raw = ask_llm(
        user_text=f"{ACTION_PLANNER_PROMPT}\n\nUser request: {user_request}",
        system_text="You convert user requests into safe structured action plans.",
        max_tokens=300,
        temperature=0
    )
    return extract_json_object(raw)

plan_action("Create an IT ticket for laptop replacement")


{'intent': 'create_it_ticket',
 'needs_tool': True,
 'tool_name': 'create_ticket',
 'arguments': {'category': 'IT Support', 'summary': 'laptop replacement'},
 'reason_for_tool': 'A ticket must be created in the backend system.'}

## Step 5 — Validate model output before execution

**Explain to learners:** Never trust raw model output blindly.

Even with structured prompting, validate:
- required keys
- allowed tool names
- argument types
- missing values
- prompt injection or unexpected tool requests

In [7]:
ALLOWED_TOOLS = {"create_ticket", "lookup_user_record", "none"}

def validate_plan(plan: dict) -> None:
    required_keys = {"intent", "needs_tool", "tool_name", "arguments", "reason_for_tool"}
    missing = required_keys - set(plan.keys())
    if missing:
        raise ValueError(f"Missing keys: {missing}")

    if plan["tool_name"] not in ALLOWED_TOOLS:
        raise ValueError(f"Disallowed tool: {plan['tool_name']}")

    if not isinstance(plan["arguments"], dict):
        raise ValueError("'arguments' must be a dictionary")

    if not isinstance(plan["needs_tool"], bool):
        raise ValueError("'needs_tool' must be a boolean")

test_plan = plan_action("Find the team and location for ananya")
validate_plan(test_plan)
test_plan


{'intent': 'lookup_user_record',
 'needs_tool': True,
 'tool_name': 'lookup_user_record',
 'arguments': {'user_name': 'ananya'},
 'reason_for_tool': 'The answer should come from the CSV file loaded from S3.'}

## Step 6 — Define backend tools

For the classroom, we keep the tools small and visible.

### What we will show
- an in-memory ticketing function
- a **CSV file lookup** function that reads a simple directory file from **S3**
- no fallback data inside the notebook

**Important:**  
In SageMaker, learners should update the bucket and file path below to point to the CSV uploaded in S3.  
For this lab, use a simple file like `training_user_directory.csv` in S3. This is safer and easier to understand than using a directory file.

### Suggested CSV columns
- `user_name`
- `full_name`
- `team`
- `location`
- `manager`
- `laptop_type`

This keeps the first-class example simple: the model decides whether to create a ticket or look up a record, and the data still comes from S3.


In [8]:
# Required S3 inputs for this lab
# Upload the CSV file to S3 and update these values before running the tool cells.
S3_DATA_BUCKET = "rag-demo-docs-sri"
S3_FILE_PATH = "training_user_directory.csv"   # example: "m1/training_user_directory.csv"

def read_csv_from_s3(bucket: str, file_path: str) -> pd.DataFrame:
    if not bucket or not file_path:
        raise ValueError("Set S3_DATA_BUCKET and S3_FILE_PATH before running this notebook.")

    obj = s3.get_object(Bucket=bucket, Key=file_path)
    return pd.read_csv(obj["Body"])

# In-memory demo store
ticket_store = []

def create_ticket(category: str, summary: str) -> str:
    ticket_id = f"TKT-{len(ticket_store)+1:03d}"
    ticket_store.append({
        "ticket_id": ticket_id,
        "category": category,
        "summary": summary
    })
    return f"Ticket created successfully: {ticket_id} | Category: {category} | Summary: {summary}"

def lookup_user_record(user_name: str) -> str:
    user_name = (user_name or "").strip().lower()
    if not user_name:
        raise ValueError("A user_name must be provided for lookup_user_record.")

    df = read_csv_from_s3(S3_DATA_BUCKET, S3_FILE_PATH)

    expected_columns = {"user_name", "full_name", "team", "location", "manager", "laptop_type"}
    missing_cols = expected_columns - set(df.columns)
    if missing_cols:
        raise ValueError(f"CSV is missing required columns: {missing_cols}")

    match_df = df[df["user_name"].astype(str).str.strip().str.lower() == user_name]

    if match_df.empty:
        return f"No record found in the S3 CSV file for user_name='{user_name}'."

    row = match_df.iloc[0]

    return (
        f"User record found in S3 file.\n"
        f"User name: {row['user_name']}\n"
        f"Full name: {row['full_name']}\n"
        f"Team: {row['team']}\n"
        f"Location: {row['location']}\n"
        f"Manager: {row['manager']}\n"
        f"Laptop type: {row['laptop_type']}"
    )


## Step 7 — Execute the prompt-to-action pipeline

This is the main classroom takeaway:
- the LLM chooses an action shape
- code validates the output
- code calls only the approved function
- the final result is logged in a structured way

In [ ]:
def execute_plan(plan: dict) -> str:
    validate_plan(plan)

    tool_name = plan["tool_name"]
    args = plan["arguments"]

    if tool_name == "create_ticket":
        return create_ticket(
            category=args.get("category", "General"),
            summary=args.get("summary", "No summary provided")
        )

    if tool_name == "lookup_user_record":
        return lookup_user_record(user_name=args.get("user_name", ""))

    return "No tool execution required."

def run_prompt_to_action_pipeline(user_request: str) -> dict:
    plan = plan_action(user_request)
    execution_result = execute_plan(plan)
    return {
        "user_request": user_request,
        "plan": plan,
        "execution_result": execution_result
    }

demo_requests = [
    "Create an IT ticket for VPN access issue",
    "Find the team and location for ananya"
]

demo_results = [run_prompt_to_action_pipeline(req) for req in demo_requests]

pipeline_log_df = pd.DataFrame([
    {
        "user_request": item["user_request"],
        "intent": item["plan"]["intent"],
        "tool_name": item["plan"]["tool_name"],
        "needs_tool": item["plan"]["needs_tool"],
        "execution_result": item["execution_result"]
    }
    for item in demo_results
])


with pd.option_context('display.max_columns', None,  # Show all columns
                       'display.max_colwidth', None, # Show full column content
                       'display.width', None):       # Auto-detect width
        display(pipeline_log_df)


,user_request,intent,tool_name,needs_tool,execution_result
0,Create an IT ticket for VPN access issue,create_it_ticket,create_ticket,True,Ticket created successfully: TKT-001 | Categor...
1,Find the team and location for ananya,lookup_user_record,lookup_user_record,True,User record found in S3 file.\nUser name: anan...


## Step 8 — Context window management

A model cannot use unlimited context.  
So enterprise systems usually:
- estimate prompt size
- chunk long text or tabular content
- select only the relevant chunks
- keep a token budget buffer for the answer

In this notebook, we will load a **simple CSV file from S3**, convert it to text, and then show a basic chunk-selection strategy.  
This helps learners see that even small business files may need careful context selection.


In [10]:
LONG_CONTEXT_S3_BUCKET = S3_DATA_BUCKET
LONG_CONTEXT_S3_FILE_PATH = S3_FILE_PATH

def load_context_text_from_s3_csv() -> str:
    df = read_csv_from_s3(LONG_CONTEXT_S3_BUCKET, LONG_CONTEXT_S3_FILE_PATH)
    return df.to_csv(index=False)

def approximate_token_count(text: str) -> int:
    # Lightweight estimate for classroom use only
    return max(1, int(len(text.split()) * 1.3))

def chunk_text(text: str, max_words: int = 120) -> List[str]:
    words = text.split()
    chunks = []
    for start in range(0, len(words), max_words):
        chunk = " ".join(words[start:start + max_words])
        chunks.append(chunk)
    return chunks

def fit_chunks_to_budget(chunks: List[str], token_budget: int = 500) -> List[str]:
    selected = []
    running_total = 0

    for chunk in chunks:
        est = approximate_token_count(chunk)
        if running_total + est > token_budget:
            break
        selected.append(chunk)
        running_total += est

    return selected

context_text = load_context_text_from_s3_csv()
chunks = chunk_text(context_text, max_words=80)
selected_chunks = fit_chunks_to_budget(chunks, token_budget=350)

print("Total chunks created:", len(chunks))
print("Chunks selected within budget:", len(selected_chunks))
print("\nPreview of first selected chunk:\n")
print(selected_chunks[0][:700] if selected_chunks else "No chunks selected.")


Total chunks created: 1
Chunks selected within budget: 1

Preview of first selected chunk:

user_name,full_name,team,location,manager,laptop_type ananya,Ananya Rao,Data Engineering,Bengaluru,Rahul Mehta,MacBook Air vikram,Vikram Shah,Platform Engineering,Hyderabad,Priya Nair,ThinkPad T14 neha,Neha Gupta,QA Automation,Pune,Sandeep Rao,MacBook Pro arjun,Arjun Iyer,IT Support,Chennai,Meera Joshi,ThinkPad X1 sara,Sara Khan,Product,Hyderabad,Dev Malik,Dell Latitude omkar,Omkar Patil,Cloud Operations,Mumbai,Ritu Sharma,MacBook Air


## Step 9 — Error handling and validation

Enterprise pipelines should fail **safely** and **clearly**.

We will capture:
- validation failures
- JSON parsing failures
- AWS API errors
- unexpected runtime errors

We will also store results in a DataFrame so the run is easy to inspect inside the notebook.

In [ ]:
def safe_run_prompt_to_action_pipeline(user_request: str) -> dict:
    try:
        result = run_prompt_to_action_pipeline(user_request)
        result["status"] = "ok"
        return result
    except (ValueError, KeyError, json.JSONDecodeError) as e:
        return {
            "status": "validation_error",
            "user_request": user_request,
            "error": str(e)
        }
    except ClientError as e:
        return {
            "status": "aws_error",
            "user_request": user_request,
            "error": str(e)
        }
    except Exception as e:
        return {
            "status": "unexpected_error",
            "user_request": user_request,
            "error": str(e)
        }

test_requests = [
    "Find the team and location for ananya",
    "Create a ticket for email access issue",
    "Find the team and location for unknown_user"
]

safe_results_df = pd.DataFrame([safe_run_prompt_to_action_pipeline(req) for req in test_requests])
with pd.option_context('display.max_columns', None,  # Show all columns
                       'display.max_colwidth', None, # Show full column content
                       'display.width', None):       # Auto-detect width
       display(safe_results_df)


,user_request,plan,execution_result,status
0,Find the team and location for ananya,"{'intent': 'lookup_user_record', 'needs_tool':...",User record found in S3 file.\nUser name: anan...,ok
1,Create a ticket for email access issue,"{'intent': 'create_it_ticket', 'needs_tool': T...",Ticket created successfully: TKT-002 | Categor...,ok
2,Find the team and location for unknown_user,"{'intent': 'lookup_user_record', 'needs_tool':...",No record found in the S3 CSV file for user_na...,ok


## Mini-hack — Extend the workflow safely

### Task
Extend the current pipeline to support one more business workflow.

### Suggested options
- reset password request
- software access request
- leave balance inquiry
- travel approval request

### Minimum success criteria
1. add one new tool
2. update the planner prompt so the tool can be selected
3. validate the new tool name
4. show one successful run and one failure-safe run

### Stretch goal
Return a final response that contains:
- action taken
- tool used
- next step for the employee

### Instructor suggestion
Ask learners to discuss:
- where validation should happen
- what should be allowlisted
- what data should come from S3
- what should never be directly controlled by the model

In [12]:
# --- LEARNER WORK AREA ---
# Add your new tool here and update the planner / validator / executor logic above.

## Wrap-up

Learners should leave this notebook understanding that:
- LLM APIs are part of software design, not just chat
- reasoning flows can be shaped through prompt structure
- prompt-to-action pipelines need validation before execution
- tool/function calling should be constrained and observable
- context window management is a design problem, not an afterthought
- secure endpoint usage starts with roles, allowlists, and minimal calls
- production readiness requires clear error handling and logging